In [ ]:
# LiitLLM — data prep
#
# Builds the corpora exactly once. Its kernel output is mounted read-only
# by every training part via kernel_sources.
#
# **Run this on a CPU session.** It is IO-bound, not GPU-bound, and CPU
# sessions do not consume the GPU quota the 20-hour training runs need.
# Doing this on a GPU session throws away hours of the budget for nothing.

In [ ]:
BAKEOFF_N = 50000
TARGET = 1000000000   # tokens per corpus
LIMIT = None          # max documents read
TOK_SAMPLE = 50000     # documents the BPE trains on
SOURCES = ("fineweb2", "hplt")   # unioned; see data.SOURCES for keep rates

In [ ]:
import glob
from pathlib import Path

def find_one(slug, filename, kind):
    """Locate a mounted source by slug. `kind` is 'datasets' or 'notebooks'.

    Kaggle mounts sources at /kaggle/input/<kind>/<owner>/<slug>/..., NOT at the
    flat /kaggle/input/<slug>/ that most examples show — a glob written for the
    flat layout silently matches nothing.

    `kind` is not optional, and that is the point. Every kernel output carries a
    full copy of the repo, so a slug-anchored search across all of /kaggle/input
    finds BOTH the repo dataset and the repo copy embedded in the previous part's
    output. Scoping the search to where the thing legitimately lives —
    the repo always in 'datasets', corpora and checkpoints always in 'notebooks'
    — makes the ambiguity impossible instead of merely detected.
    """
    hits = glob.glob(f"/kaggle/input/{kind}/**/{slug}/**/{filename}", recursive=True)
    assert len(hits) == 1, (
        f"expected exactly 1 {filename} under {kind}/{slug}, found {hits}.\n"
        f"Sources actually mounted: {sorted(glob.glob('/kaggle/input/*/*/*'))}"
    )
    return Path(hits[0])

In [ ]:
import shutil, os, sys

REPO_SLUG = "liitllm-repo"   # dataset holding this repo
PREP_SLUG = "00-prep"        # kernel whose OUTPUT holds the corpora + tokenizer

repo_src = find_one(REPO_SLUG, "pyproject.toml", "datasets").parent
REPO = Path("/kaggle/working/liitllm-repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(repo_src, REPO)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
print(f"repo: {repo_src} -> {REPO}")

In [ ]:
!pip install -q datasets tokenizers

In [ ]:
# Which corpus actually preserved code-switching? Standard pipelines filter
# it out, so this measures how much each one threw away — the answer is a
# case-study figure either way.
from liitllm import data as D
D.bakeoff(n=BAKEOFF_N)

In [ ]:
# Build both corpora in one pass: same source, same order, same dedup, the
# only difference is the Taglish filter. `prepare` asserts they finish at
# identical token counts — if they differ, the ablation would be measuring
# corpus SIZE rather than corpus content, and that looks exactly like a result.
#
# Set sources from the bakeoff above.
D.prepare(target=TARGET, vocab_size=8192, limit=LIMIT,
          tok_sample=TOK_SAMPLE, sources=SOURCES)

In [ ]:
# Sanity-check the filter by eye before spending 40 GPU-hours on its output.
# A bad lexicon shifts every score without ever raising an error.
from liitllm import taglish as tg
import itertools
for doc in itertools.islice(D._stream(SOURCES[0]), 200):
    tl, en = tg.score(doc)
    mark = "KEEP" if tg.is_taglish(doc) else "drop"
    print(f"[{mark} tl={tl:.2f} en={en:.2f}] {doc[:160]}")

In [ ]:
# Stage the corpora at the TOP of /kaggle/working so they become this
# kernel's output, which the training parts mount via kernel_sources. No API
# credentials and no dataset versioning: the artifacts move as kernel output.
#
# The repo copy is removed first — every kernel output otherwise carries a
# full copy of the repo, which is what makes a bare glob for pyproject.toml
# resolve to a stale snapshot in later sessions.
import shutil
stage = Path('/kaggle/working/data')
if stage.exists():
    shutil.rmtree(stage)
shutil.copytree(REPO / 'data', stage)
shutil.rmtree(REPO, ignore_errors=True)
for p in sorted(stage.rglob('*')):
    if p.is_file():
        print(f'{p.relative_to(stage)}  {p.stat().st_size / 1e6:.1f} MB')